# Module 1: Advanced DAX Concepts

## Week 11 | June 16, 2026

Week 10 covered the two languages: M (Power Query, before-the-model cleaning and shaping) and DAX (after-the-model analysis and calculations). Module 1 this week goes deeper into DAX with explicit filtering (`CALCULATE()`, `FILTER()`, `ALL()`), calculated tables, iterators (the X-functions), and Row-Level Security. These are the tools that let you control exactly what data a formula sees and who can see what in a published report.

This module covers four sections: 1.A Filtering, 1.B Calculated Tables, 1.C Iterators, and 1.D Row Level Security.

## By End of Module 1 I Should Be Able To:

- [ ] Explain the difference between implicit and explicit filtering in DAX
- [ ] Use `CALCULATE()` to override filter context with a simple single-column condition
- [ ] Use `FILTER()` to apply conditions that span multiple columns
- [ ] Use `ALL()` to remove implicit filters and calculate ratios against a whole dataset
- [ ] Create a calculated table using a DAX expression and explain when to use one
- [ ] Explain what an iterator is and how row context works inside one
- [ ] Identify common X-functions and when to use them vs a simple aggregation
- [ ] Explain evaluation context (row context vs filter context)
- [ ] Define RLS roles in Desktop, test them with View As, and understand how users are assigned in Power BI Service

**Resource**: https://www.datacamp.com/tutorial/power-bi-dax-tutorial-for-beginners


<h2 style="color:steelblue">Section 1.A: Filtering</h2>

**What Is Filtering in DAX?**

Every DAX calculation runs inside a **filter context**, which is the set of rules that determines which rows are included when a formula evaluates. Filtering is what shapes that context. You use filtering any time you need to limit a calculation to a subset of data (e.g. only East Region sales), drill down from a high-level summary into subcategories, or compare time periods (this year vs last year, YTD, MTD). There are two kinds of filtering: **implicit** and **explicit**.

**Implicit vs Explicit Filtering**

These are the two ways filter context gets applied in Power BI.

| Type | Who applies it | How it works |
|------|---------------|---------------|
| **Implicit** | The report layout or the user | Automatically applied based on visuals and user interactions |
| **Explicit** | The DAX formula itself | Written directly into the formula using `CALCULATE()`, `FILTER()`, or `ALL()` |

**Implicit filtering** happens automatically. When you add Region to a bar chart, Power BI creates a separate filter context for each bar — each bar only "sees" its own region's rows. When you click a pie slice, that filter extends to every other visual on the page. For example: in a table showing Sales Amount by country, the value shown for Argentina ($480) is already implicitly filtered — `SUM(SalesAmount)` only adds up rows where Country = Argentina. If you also click the Condiments slice in a pie chart, both filters stack: now Argentina AND Condiments must both be true.

**Explicit filtering** overrides implicit filtering on a column-by-column basis — the DAX formula itself decides the filter conditions, regardless of what the user selected in the report.

> **Pandas analogy:** Implicit filtering is like `.groupby()` — each group automatically only sees its own rows. Explicit filtering is like writing `df[df['Region'] == 'East']` directly inside your calculation, no matter what the rest of the code is doing.

**DAX Syntax — Fully Qualified Column Names**

Before writing any explicit filter functions, one important syntax rule: always reference columns as `TableName[ColumnName]`. This is called a **fully qualified column name**, and it's best practice even when referencing a column in the same table the formula lives in. If a table name contains spaces, reserved keywords, or special characters outside the ANSI alphanumeric range, wrap the table name in single quotes: `'My Table'[Column]`.

Example breakdown:

    Total Sales = SUM(Sales[SalesAmount])

| Part | What it is |
|------|------------|
| `Total Sales` | Measure name |
| `=` | Starts the formula |
| `SUM` | DAX function |
| `Sales` | Referenced table |
| `[SalesAmount]` | Referenced column |

**`CALCULATE()` — Fastest Way to Explicitly Filter**

`CALCULATE()` lets you evaluate an expression under a modified filter context. It's the most common explicit filter function in DAX.

Syntax:

    CALCULATE(<expression>, <filter1>, <filter2>, ...)

Important rule: each filter argument must reference a **single column** with a simple comparison. If your condition involves combining or calculating across multiple columns, `CALCULATE()` alone won't work — you need `FILTER()` for that.

Works — single column condition:

    Sum Green Products Sold = CALCULATE(
        SUM(Sales[Total]),
        [Color] = "Green"
    )

    Average Freight for High Value Orders = CALCULATE(
        AVERAGE(Sales[FreightAmount]),
        Sales[SaleAmount] > 10
    )

Does NOT work — multi-column expression in the condition:

    -- This fails because the filter combines two columns:
    CALCULATE(
        AVERAGE(Sales[FreightAmount]),
        Sales[UnitPrice] * (1 - [DiscountPercent]) > 10
    )

> **Why it's fast:** Power BI stores each column's data together in memory (columnar storage). `CALCULATE()` takes advantage of this — a single-column filter is extremely efficient to evaluate.

**`FILTER()` — For Multi-Column or Complex Conditions**

`FILTER()` is used when your filter condition needs to look at more than one column at a time, or when you need row-level logic that `CALCULATE()` can't handle alone.

Syntax:

    FILTER(<table>, <condition>)

It takes a **table** as input and an **expression** that gets evaluated for each row. It returns a **table** containing only rows where the condition is true. Because the result is a table, it can be used anywhere a table value is expected — including inside `CALCULATE()`.

Simple example — single column:

    Products = FILTER(
        'Products',
        'Products'[BrandName] = "GoodForYou"
    )

Multi-column example — where `FILTER()` shines:

    HighMarginProducts = FILTER(
        'Products',
        AND(
            'Products'[BrandName] = "GoodForYou",
            'Products'[UnitPrice] > 'Products'[UnitCost] * 2
        )
    )

Using `FILTER()` inside `CALCULATE()`:

    High Margin Sales = CALCULATE(
        SUM(Sales[Total]),
        FILTER('Products', 'Products'[UnitPrice] > 'Products'[UnitCost] * 2)
    )

Heads up: nesting `FILTER()` inside `CALCULATE()` is powerful but significantly slows down the calculation. Use it only when a simpler single-column condition won't work.

> **Pandas analogy:** `FILTER()` is like `df[condition]` — it creates a filtered version of a table. The difference is that in DAX, the result stays as a table object that gets passed into another function.

**`ALL()` — Remove Implicit Filters**

`ALL()` clears implicit filters so your formula can see the full dataset, regardless of what's selected in the report. It can be applied to a specific column (`ALL(Sales[Region])` removes the filter on Region only) or to an entire table (`ALL(Sales)` removes all filters on the Sales table).

Most common use case — calculating a percentage of total:

    SalesPercent = DIVIDE(
        SUMX(Sales, Sales[Quantity] * Sales[NetPrice]),
        SUMX(ALL(Sales), Sales[Quantity] * Sales[NetPrice])
    )

What this does: the numerator uses the current filter context (e.g. one region), while the denominator uses `ALL(Sales)` to ignore that filter and get the company-wide total. The result is the selected region's share of total sales.

> **Pandas analogy:** `ALL()` is like temporarily using the full `df` in your denominator while the numerator uses a filtered `df_region`. You'd have to manually track both DataFrames — DAX handles it in one expression.

### Confusion Corner — Section 1.A

**Why can't `CALCULATE()` handle multi-column conditions directly?** Because `CALCULATE()` is optimized for columnar storage — it evaluates one column at a time very efficiently. Multi-column conditions require row-by-row evaluation, which is what `FILTER()` is built for. That's why you combine them: `CALCULATE()` sets up the context, `FILTER()` handles the complex row-level logic.

**Does `ALL()` permanently remove filters?** Only within the scope of that formula. It doesn't change what users see or affect other visuals. It just tells that specific DAX expression to ignore the current filter context when it runs.

**What's the difference between `FILTER()` returning a table and a measure returning a number?** `FILTER()` on its own produces a table — you can't display a table as a single value in a card visual. You have to wrap it in an aggregation like `SUM()`, `COUNTROWS()`, or another function that turns the table into a scalar (single number).

<h2 style="color:steelblue">Section 1.B: Calculated Tables</h2>

**What Is a Calculated Table?**

A calculated table is the third of four DAX calculation types (alongside calculated columns, measures, and row filters). It's a new table you create inside the data model using a DAX expression — not by importing from a data source. The DAX expression must return a **table data type**, so functions like `FILTER()`, `UNION()`, `NATURALINNERJOIN()`, and `DATATABLE()` can all be used.

When to use a calculated table: when you'd otherwise repeat the same `FILTER()` expression across many different measures; when you need a "role-playing" table (covered in Module 2 with date tables); or when you want to combine or flatten data from multiple tables without going back to Power Query.

> **Pandas analogy:** A calculated table is like creating a new DataFrame from existing ones using `pd.concat()`, `pd.merge()`, or building a dict — except it lives inside the Power BI model and stays connected to the rest of your relationships.

**Calculated Table Examples**

Three common patterns:

Combining two tables (like SQL UNION):

    Global Sales = UNION(NorthAmericanSales, EuropeanSales)

Joining two tables (like SQL INNER JOIN):

    Product Categories = NATURALINNERJOIN(Categories, Subcategories)

This is an alternative to flattening data in Power Query when you need to bring over many columns from two related tables.

Hard-coding a lookup table with `DATATABLE()`:

    Order Status = DATATABLE(
        "OrderStatusId", INTEGER,
        "Status", STRING,
        {
            {1, "In Cart"},
            {2, "Placed"},
            {3, "Acknowledged"},
            {4, "Shipped"},
            {5, "Delivered"},
            {6, "Cancelled"}
        }
    )

Useful when you need a small reference table that doesn't exist in your data source.

### Confusion Corner — Section 1.B

**Is a calculated table the same as a calculated column?** No. A calculated column adds a new column to an existing table, row by row. A calculated table creates an entirely new table in the model.

**When would I use a calculated table vs doing the same thing in Power Query?** Power Query is better when you want to clean or reshape raw source data before it enters the model. A calculated table is better when you need to derive a new table from data that's already in the model — for example, combining two fact tables that are both already loaded.

<h2 style="color:steelblue">Section 1.C: Iterators</h2>

**What Is an Iterator?**

An iterator is a special class of DAX function that loops through a table **row by row**, evaluates an expression for each row, and then does something with the results. Most iterator functions end in the letter **X** (e.g. `SUMX`, `AVERAGEX`, `RANKX`).

Syntax pattern:

    ITERATORNAME(<table>, <expression per row>)

Example — `SUMX()`:

    Total Revenue = SUMX(
        Sales,
        Sales[NetPrice] * Sales[Quantity]
    )

What happens: for every row in Sales, DAX multiplies `NetPrice` by `Quantity`. Then it adds all those individual results together into a single number. You can't do this with `SUM()` because `SUM()` only aggregates an existing column — it can't calculate row-by-row first.

> **Pandas analogy:** `SUMX()` is like `(df['NetPrice'] * df['Quantity']).sum()` — you multiply column by column first (row-level), then sum the result. Plain `SUM()` in DAX is more like `df['Revenue'].sum()` — it only works if the column already exists.

**Common Iterator Functions**

Not all iterators return the same kind of result. `FILTER()` is also an iterator — it loops row by row and tests a condition, but returns a table instead of a single value.

| Iterator | Returns | What it does |
|----------|---------|---------------|
| `SUMX()` | Scalar (number) | Sums a row-level expression across a table |
| `AVERAGEX()` | Scalar | Averages a row-level expression |
| `MINX()` / `MAXX()` | Scalar | Finds the min or max of a row-level expression |
| `COUNTX()` | Scalar | Counts non-blank results of a row-level expression |
| `COUNTAX()` | Scalar | Counts non-blank values including text |
| `RANKX()` | Scalar | Ranks a value within a table |
| `CONCATENATEX()` | Text | Concatenates text values row by row |
| `PRODUCTX()` | Scalar | Multiplies values row by row |
| `FILTER()` | Table | Returns only the rows where the condition is true |

**Evaluation Context — Row Context vs Filter Context**

Evaluation context is the environment in which a DAX formula runs. It has two parts that can both be active at the same time.

| Context | What it is | Where it applies | Key behaviors |
|---------|-----------|------------------|---------------|
| **Row Context** | Limits the formula to the current row | Inside iterators and calculated columns | Does NOT propagate across relationships. Use `RELATED()` or `RELATEDTABLE()` to access other tables. |
| **Filter Context** | Limits the formula to rows matching active filters | Measures, report visuals, DAX filters | Can be changed with `CALCULATE()`, cleared with `ALL()`. Propagates across relationships automatically. |

When you're inside a `SUMX()`, you have row context (you can see the current row's values). But the overall filter context from the report (e.g. the user selected East Region) still applies on top — both are active at the same time.

**Nesting Iterators**

Iterators can be nested inside each other. Think of it as writing steps from the inside out.

Example — ranking products by total sales:

    ProductSalesRank =
    RANKX(
        ALL(Products),
        SUMX(
            RELATEDTABLE(InternetSales),
            'InternetSales'[SalesAmount]
        )
    )

The inner `SUMX()` calculates total sales for each product using `RELATEDTABLE()` to follow the relationship. The outer `RANKX()` ranks all products against each other using `ALL(Products)` to ignore the current filter context.

Performance warning: nesting iterators works but can be slow at scale. Before nesting, ask if a simpler approach exists. In this example, if a `[Total Sales]` measure already exists, this is cleaner:

    ProductSalesRank = RANKX(ALL(Products), [Total Sales])

The student guide specifically warns that an AI tool asked to "rank products by sales" might generate the nested version without knowing that a simpler measure already exists in the model.

### Confusion Corner — Section 1.C

**Why do I need `RELATED()` inside an iterator?** Row context doesn't automatically cross table relationships. If you're iterating over Sales and need a value from Products, you need `RELATED(Products[Category])` to follow the relationship for the current row.

**What's `RELATEDTABLE()` for?** The reverse — instead of going from the many-side to the one-side (which `RELATED()` does), `RELATEDTABLE()` goes from the one-side to the many-side and returns all related rows as a table. That's why it's commonly wrapped in an iterator: `SUMX(RELATEDTABLE(Sales), Sales[Amount])`.

**When do I use `SUM()` vs `SUMX()`?** Use `SUM()` when the column already has the values you want to add up. Use `SUMX()` when you need to calculate something per row first (like multiplying two columns together) before summing.

<h2 style="color:steelblue">Section 1.D: Row Level Security</h2>

**What Is Row Level Security (RLS)?**

RLS restricts which rows a user can see in a report based on their identity. Instead of building separate reports for different user groups, you build one report and let RLS control what each person sees. Filters are defined within **roles**, and each role has a DAX expression that acts as a row-level filter — only rows where the expression evaluates to true are visible. Roles are defined in **Power BI Desktop**, but users are assigned to roles in **Power BI Service**.

Example: a role called `East Region` with the filter `[Region] = "East"` means anyone assigned to that role only sees East Region data across every visual in the report.

> **Real-world use case:** A company with regional sales managers. Each manager should only see their own region's data, even though the underlying dataset has all regions.

**Static vs Dynamic RLS**

There are two approaches to setting up roles, and the right choice depends on how many users and regions you're dealing with.

| Comparison Area | Static RLS | Dynamic RLS |
|----------------|------------|-------------|
| Security Logic | Hard-coded DAX filter: `[Region] = "East"` | Uses user identity + security table to determine access automatically |
| Number of Roles | One role per region/group (East, West, South...) | Typically only one role for the whole organization |
| Maintenance | High — must update roles as users and regions change | Low — just update the security table |
| Scalability | Best for small orgs with few users | Best for large orgs with many users/departments |
| User Access | Users manually assigned to each role | Access determined automatically via `USERPRINCIPALNAME()` |

**Defining and Testing RLS**

To set up a role in Power BI Desktop:

1. Go to **Modeling** tab (Report view) or **Home** tab (Table/Model view) and select **Manage Roles**
2. Click **New** to create a new role
3. Give it a name (e.g. `East Region`)
4. Select the table to filter on
5. Enter a DAX expression in the DAX Editor, or use the Column/Condition/Value builder
6. Click **Save**

To test: select **View as** in the ribbon, choose the role(s) to preview, click OK, then browse the report pages to confirm the filter applies everywhere.

Users are assigned in **Power BI Service** under the Security settings for the semantic model (not the report). Users can be added one at a time by email, or larger distribution list emails can add whole groups.

### Confusion Corner — Section 1.D

**Why can't I assign users to roles in Desktop?** Desktop is a local development tool — it doesn't know who your actual users are. User assignment is a deployment decision that happens in Power BI Service where the actual user accounts exist.

**Does RLS affect admins?** Workspace admins and report owners can see all data regardless of RLS when accessing semantic model settings. RLS applies to users who are explicitly assigned to roles.

**What's the difference between RLS and just filtering the report?** A report filter is visual-only — a user can remove it or ignore it. RLS is enforced at the data level — the user literally cannot see the restricted rows no matter what they do in the report.

## When Would I Use This?

- **`CALCULATE()`** — any time you need a measure that ignores or overrides what the user selected. Example: "show the East total even when the West slicer is selected."
- **`FILTER()`** — when your filter condition involves comparing two columns or applying AND/OR logic across multiple columns.
- **`ALL()`** — percentage-of-total calculations, or any time a measure needs the full dataset regardless of current filters.
- **Calculated tables** — when you need a new table derived from model data (combining regions, creating a status lookup), especially when the same logic would repeat across many measures.
- **Iterators (`SUMX`, etc.)** — when you need row-level math before aggregating. If you're multiplying two columns before summing, you need an iterator.
- **RLS** — when different users should see different subsets of data in the same report. Common in sales, HR, finance.

## Key Takeaways — Module 1

- Implicit filtering is automatic (report layout and user interaction). Explicit filtering is written into DAX formulas.
- `CALCULATE()` is the go-to for explicit filtering — fast, simple, single-column conditions only.
- `FILTER()` handles complex multi-column conditions — returns a table, so wrap it in an aggregation or use inside another function.
- `ALL()` removes implicit filters so a formula can see the full dataset — essential for ratio/percentage calculations.
- Calculated tables are DAX-generated tables inside the model — useful for combining data and avoiding repeated filter logic.
- Iterators loop row by row: they need a table and a per-row expression. Most end in X. They give you row context inside the loop.
- Row context = current row (inside iterators and calculated columns). Filter context = active report/DAX filters on the full model.
- RLS restricts data visibility by user role. Defined in Desktop with DAX, assigned to users in Power BI Service.